# 02c · V2: the improved ResNet-50 on all folds

Uses the configuration chosen in 02b (`results/cnn/v2_config.json`). Same folds, same evaluation as every other
model. Time: about 6-8 min per fold (≈ 35 min). Finished folds are skipped on re-run.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "PLAN.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import json

import pandas as pd

from common import config, data, env
from common.cv import run_cv
from common.train import TrainConfig

config.ensure_dirs()
device = env.require_cuda()
env.seed_everything()
cfg_file = config.CNN_RESULTS / "v2_config.json"
if not cfg_file.exists():
    raise FileNotFoundError("Run 02b_resnet50_v2_ablation.ipynb first.")
V2 = TrainConfig(**json.loads(cfg_file.read_text()))
print(V2)
uniq, splits = data.load_unique(), data.load_splits()
images = data.load_images(uniq)

In [ ]:
summary = run_cv(V2, device, uniq, splits, images)

In [ ]:
pd.DataFrame({k: {"best_epoch": v["best_epoch"], "minutes": round(v["train_minutes"], 1),
                  "test_acc": v["test"]["accuracy"], "test_macro_f1": v["test"]["macro_f1"]}
              for k, v in summary["folds"].items()}).T